<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_02_data_and_sampling_distributions/02_data_and_sampling_distributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 2 folder
%cd chapter_02_data_and_sampling_distributions

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 148 (delta 88), reused 72 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 515.10 KiB | 3.37 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_02_data_and_sampling_distributions


# Chapter 02: Data and Sampling Distributions

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 2: Data and Sampling Distributions

## Goals

In this notebook, I will:
- Understand populations and samples
- Explore random sampling
- Identify sampling bias and selection bias
- Understand regression to the mean
- Distinguish between data distribution and sampling distribution
- Build intuition for the Central Limit Theorem (CLT)
- Calculate and interpret standard error
- Use the bootstrap to estimate sampling distributions
- Construct and interpret confidence intervals
- Recognize normal, t, binomial, Poisson, and other distributions
- Connect sampling concepts with machine learning

---

## Topics Covered

- Population vs sample
- Random sampling
- Sampling bias and selection bias
- Regression to the mean
- Sampling distributions
- Central Limit Theorem (CLT)
- Standard error
- Bootstrap
- Confidence intervals
- Normal distribution and QQ-plots
- Student's t-distribution
- Binomial distribution
- Poisson and exponential distributions
- ML relevance and applications

---

## 🤖 ML Relevance

> Why does sampling theory matter for Machine Learning?
> * **Train/Test Splits**: Understanding sampling ensures your validation strategy is sound
> * **Bootstrap Methods**: Used in ensemble methods like bagging and random forests
> * **Confidence Intervals**: Quantify uncertainty in model predictions and metrics
> * **Distribution Assumptions**: Many ML algorithms assume normality or specific distributions
> * **Bias Detection**: Sampling theory helps identify and correct dataset biases

---

## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats
    from scipy.stats import trim_mean, binom, norm, t, chi2, poisson, expon, sem
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Helper function to load book datasets
def load_book_data(filename):
    """Load data from local dir or fetch from GitHub if missing."""
    local_path = f'../datasets/raw/{filename}'
    if os.path.exists(local_path):
        return pd.read_csv(local_path)
    else:
        base_url = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"
        print(f"Downloading {filename} from book repo...")
        return pd.read_csv(base_url + filename)

# Create directories if needed
os.makedirs('../datasets/raw', exist_ok=True)
os.makedirs('../datasets/processed', exist_ok=True)

print("Notebook setup complete")
```
</details>



---

## Load Dataset

<details>
<summary>Click to view solution</summary>

```python
state = pd.read_csv("../datasets/raw/state.csv") if os.path.exists("../datasets/raw/state.csv") else pd.read_csv(
    "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/state.csv"
)

state.head()
```
</details>



---

# Section 1: Population vs Sample

## What is a population?

A population represents the **entire group of interest**.

Example:
- All airline passengers
- All customers
- All flights
- All states

In our dataset, all states = population. But in real life, analysing the entire population is often expensive or impossible. So we use samples.

<details>
<summary>Click to view solution</summary>

```python
print(f"Population size: {len(state)}")
```
</details>


## What is a sample?

A sample is a subset of the population. Good samples should represent the population fairly.

<details>
<summary>Click to view solution</summary>

```python
sample = state.sample(n=10, random_state=42)
sample
```
</details>

<details>
<summary>Click to view solution</summary>

```python
print("Population Mean:", f"{state['Population'].mean():,.2f}")
print("Sample Mean:", f"{sample['Population'].mean():,.2f}")
```
</details>

## Reflection

Questions:
1. Why are the means different?
2. Can a sample still represent a population?
3. What happens if the sample is too small?



---

# Section 2: Random Sampling

## Why random sampling matters

Good sampling requires every observation should have equal chance. Why? Because non-random samples create bias.

<details>
<summary>Click to view solution</summary>

```python
random_sample = state.sample(frac=0.30, random_state=42)
print(f"Random sample size: {len(random_sample)}")
random_sample.head()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
print("Population Mean:", f"{state['Population'].mean():,.2f}")
print("Random Sample Mean:", f"{random_sample['Population'].mean():,.2f}")
```
</details>

## Reflection

Questions:
1. Are sample statistics close to population statistics?
2. Why are they not identical?
3. Why is randomness important?


---

# Section 3: Sampling Bias

## What is sampling bias?

Bias occurs when some observations are favoured. Suppose we only sampled large states — would this represent the US properly?

### The Literary Digest Poll of 1936

A classic example of sample bias: polling only affluent voters predicted Landon over Roosevelt.

<details>
<summary>Click to view solution</summary>

```python
# Simulate the Literary Digest scenario
np.random.seed(42)

# True population: 60% Roosevelt, 40% Landon
true_population = np.random.choice(['Roosevelt', 'Landon'], size=100000, p=[0.6, 0.4])

# Biased sample: only affluent voters (who favored Landon)
affluent_sample = np.random.choice(['Roosevelt', 'Landon'], size=2000000, p=[0.4, 0.6])

# Random sample (what Gallup did)
random_sample_poll = np.random.choice(true_population, size=2000, replace=False)

# Calculate results
print("True Population Result:")
print(f"  Roosevelt: {np.mean(true_population == 'Roosevelt')*100:.1f}%")
print(f"  Landon: {np.mean(true_population == 'Landon')*100:.1f}%")

print("\nLiterary Digest (Biased Sample):")
print(f"  Roosevelt: {np.mean(affluent_sample == 'Roosevelt')*100:.1f}%")
print(f"  Landon: {np.mean(affluent_sample == 'Landon')*100:.1f}%  <- Wrong prediction!")

print("\nGallup (Random Sample, n=2000):")
print(f"  Roosevelt: {np.mean(random_sample_poll == 'Roosevelt')*100:.1f}%  <- Correct prediction")
print(f"  Landon: {np.mean(random_sample_poll == 'Landon')*100:.1f}%")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
categories = ['Roosevelt', 'Landon']
x = np.arange(len(categories))

axes[0].bar(x, [np.mean(true_population == c) for c in categories], color='skyblue')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].set_ylabel('Proportion')
axes[0].set_title('True Population')
axes[0].set_ylim(0, 1)

axes[1].bar(x, [np.mean(affluent_sample == c) for c in categories], color='salmon')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel('Proportion')
axes[1].set_title('Biased Sample (Literary Digest)')
axes[1].set_ylim(0, 1)

axes[2].bar(x, [np.mean(random_sample_poll == c) for c in categories], color='lightgreen')
axes[2].set_xticks(x)
axes[2].set_xticklabels(categories)
axes[2].set_ylabel('Proportion')
axes[2].set_title('Random Sample (Gallup)')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()
```
</details>


### Biased Sampling with State Data

<details>
<summary>Click to view solution</summary>

```python
biased_sample = state.nlargest(10, "Population")
biased_sample
```
</details>

<details>
<summary>Click to view solution</summary>

```python
print("Population Mean:", f"{state['Population'].mean():,.2f}")
print("Biased Sample Mean:", f"{biased_sample['Population'].mean():,.2f}")

comparison = pd.DataFrame({
    "Population": state["Population"],
    "Biased Sample": biased_sample["Population"]
})

comparison.plot(kind="box", figsize=(8, 5))
plt.title("Population vs Biased Sample")
plt.show()
```
</details>

## Common Forms of Selection Bias

| Type | Description | Example |
|------|-------------|---------|
| **Data Snooping** | Hunting through data until something "interesting" appears | Testing 20 features, reporting only the "significant" one |
| **Vast Search Effect** | Repeated modeling increases false discoveries | Running hundreds of A/B tests, reporting only the winner |
| **Cherry-Picking** | Selecting time periods that support a conclusion | Showing only Q4 results when annual trend is negative |
| **Self-Selection** | Participants choose to respond, creating non-representative sample | Online reviews skewed toward extreme experiences |

## Reflection

Questions:
1. Why is the biased sample misleading?
2. Does more data always mean better data?
3. Can large datasets still be biased?



---

# Section 4: Regression to the Mean

Extreme observations tend to be followed by more central ones.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Simulate performance = skill + luck
n_players = 1000
skill = np.random.normal(0, 1, n_players)  # True skill (stable)
luck_year1 = np.random.normal(0, 2, n_players)  # Luck in year 1 (variable)
luck_year2 = np.random.normal(0, 2, n_players)  # Luck in year 2 (variable)

performance_year1 = skill + luck_year1
performance_year2 = skill + luck_year2

# Select "top performers" from year 1 (top 10%)
top_10_pct = np.percentile(performance_year1, 90)
top_performers = performance_year1 >= top_10_pct

# Compare their year 2 performance
print("Top 10% performers in Year 1:")
print(f"  Year 1 avg performance: {np.mean(performance_year1[top_performers]):.2f}")
print(f"  Year 2 avg performance: {np.mean(performance_year2[top_performers]):.2f}")
print(f"  Regression toward mean: {np.mean(performance_year1[top_performers]) - np.mean(performance_year2[top_performers]):.2f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Year 1 vs Year 2 for top performers
axes[0].scatter(performance_year1[top_performers], performance_year2[top_performers],
                alpha=0.6, edgecolors='black')
axes[0].plot([performance_year1.min(), performance_year1.max()],
             [performance_year1.min(), performance_year1.max()],
             'r--', label='Perfect correlation')
axes[0].set_xlabel('Year 1 Performance')
axes[0].set_ylabel('Year 2 Performance')
axes[0].set_title('Top 10% Performers: Year 1 vs Year 2')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Distribution comparison
axes[1].hist(performance_year1[top_performers], bins=20, alpha=0.5, label='Year 1', density=True)
axes[1].hist(performance_year2[top_performers], bins=20, alpha=0.5, label='Year 2', density=True)
axes[1].axvline(np.mean(performance_year1[top_performers]), color='blue', linestyle='--', label='Year 1 Mean')
axes[1].axvline(np.mean(performance_year2[top_performers]), color='green', linestyle='--', label='Year 2 Mean')
axes[1].set_xlabel('Performance')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution Shift: Regression to Mean')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nML Relevance: When evaluating model improvements, ensure gains aren't just regression to mean!")
```
</details>


---

# Section 5: Sampling Variability

## Key idea

Every sample is different. Even random samples produce slightly different statistics. This variation is called sampling variability.

<details>
<summary>Click to view solution</summary>

```python
sample_means = []

for i in range(1000):
    temp_sample = state.sample(n=10, replace=True)
    sample_means.append(temp_sample["Population"].mean())
```
</details>

<details>
<summary>Click to view solution</summary>

```python
plt.figure(figsize=(10, 6))
sns.histplot(sample_means, bins=30, kde=True)
plt.title("Distribution of Sample Means")
plt.xlabel("Sample Mean")
plt.show()
```
</details>

## Reflection

Questions:
1. Why are means different?
2. Why do they form a distribution?
3. Why does this matter?


---

# Section 6: Sampling Distribution of a Statistic

## Key Distinction

| Concept | Definition | Example |
|---------|-----------|---------|
| **Data Distribution** | Distribution of individual values in a dataset | Histogram of loan amounts |
| **Sampling Distribution** | Distribution of a statistic over many samples | Distribution of sample means from repeated sampling |

---



# Section 7: Central Limit Theorem (CLT)

## One of the most important ideas in statistics

The CLT says: As sample size increases, sample means become approximately normal — even if original data is skewed.

<details>
<summary>Click to view solution</summary>

```python
plt.figure(figsize=(10, 6))
sns.histplot(state["Population"], bins=20, kde=True)
plt.title("Original Population Distribution")
plt.show()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Generate sampling distribution for different sample sizes
np.random.seed(42)

# Create a highly skewed population (exponential distribution)
population = np.random.exponential(scale=2, size=100000)

def sampling_distribution(population, sample_size, n_samples=1000):
    sample_means = []
    for _ in range(n_samples):
        sample = np.random.choice(population, size=sample_size, replace=False)
        sample_means.append(np.mean(sample))
    return np.array(sample_means)

# Generate sampling distributions for different sample sizes
sample_sizes = [1, 5, 20, 50]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, n in enumerate(sample_sizes):
    sample_means = sampling_distribution(population, sample_size=n)
    
    axes[idx].hist(sample_means, bins=30, density=True, alpha=0.7, edgecolor='black')
    
    # Add normal curve for comparison
    mu, sigma = np.mean(sample_means), np.std(sample_means)
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
    axes[idx].plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal approximation')
    
    axes[idx].axvline(np.mean(population), color='green', linestyle='--', label=f'Population mean: {np.mean(population):.2f}')
    axes[idx].set_xlabel('Sample Mean')
    axes[idx].set_ylabel('Density')
    axes[idx].set_title(f'Sampling Distribution (n={n})\nSE = {np.std(sample_means):.3f}')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(alpha=0.3)

plt.suptitle('Central Limit Theorem: Sample Means Approach Normality', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Print standard error comparison
print("\nStandard Error vs Sample Size:")
print(f"  Population SD: {np.std(population):.3f}")
for n in sample_sizes:
    se_theoretical = np.std(population) / np.sqrt(n)
    se_empirical = np.std(sampling_distribution(population, sample_size=n))
    print(f"  n={n:2d}: Theoretical SE={se_theoretical:.3f}, Empirical SE={se_empirical:.3f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Compare different sample sizes side by side
means_5 = []
means_30 = []
means_50 = []

for _ in range(1000):
    means_5.append(state.sample(5, replace=True)["Population"].mean())
    means_30.append(state.sample(30, replace=True)["Population"].mean())
    means_50.append(state.sample(50, replace=True)["Population"].mean())

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(means_5, kde=True, ax=ax[0])
ax[0].set_title("Sample Size = 5")

sns.histplot(means_30, kde=True, ax=ax[1])
ax[1].set_title("Sample Size = 30")

sns.histplot(means_50, kde=True, ax=ax[2])
ax[2].set_title("Sample Size = 50")

plt.show()
```
</details>

## Key takeaway

Larger sample size = more stable estimates. This is why more representative data often beats more complicated models in machine learning.



---

# Section 8: Standard Error

## What is Standard Error?

Standard error measures uncertainty in a sample estimate. It tells us: "How much would our estimate change if we sampled again?"

Difference:
- **Standard deviation**: Measures spread of observations
- **Standard error**: Measures spread of estimates

Smaller standard error means more reliable estimate.

### The Square Root of n Rule

$$SE = \frac{s}{\sqrt{n}}$$

To halve SE, quadruple sample size.

<details>
<summary>Click to view solution</summary>

```python
population_std = state["Population"].std()
sample_size = len(state)
standard_error = population_std / np.sqrt(sample_size)

print(f"Standard Deviation: {population_std:,.2f}")
print(f"Sample Size: {sample_size}")
print(f"Standard Error: {standard_error:,.2f}")

# Using scipy
sem_value = sem(state["Population"])
print(f"Scipy Standard Error: {sem_value:,.2f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Demonstrate SE vs sample size with loan income data
try:
    loans_income = load_book_data('loans_income.csv')['income']
except:
    loans_income = np.random.lognormal(mean=10, sigma=1, size=10000)

# Calculate SE for different sample sizes
sample_sizes = [25, 100, 400, 1600]
results = []

for n in sample_sizes:
    sample_means = [np.mean(np.random.choice(loans_income, size=n, replace=True))
                    for _ in range(1000)]
    se_empirical = np.std(sample_means)
    se_theoretical = np.std(loans_income) / np.sqrt(n)
    
    results.append({
        'n': n,
        'empirical_se': se_empirical,
        'theoretical_se': se_theoretical,
        'ratio': se_empirical / se_theoretical
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Visualize the square root relationship
plt.figure(figsize=(8, 6))
plt.plot(results_df['n'], results_df['empirical_se'], 'bo-', label='Empirical SE', markersize=8)
plt.plot(results_df['n'], results_df['theoretical_se'], 'r--', label='Theoretical SE: sigma/sqrt(n)', linewidth=2)
plt.xlabel('Sample Size (n)')
plt.ylabel('Standard Error')
plt.title('Standard Error Decreases as 1/sqrt(n)')
plt.xscale('log')
plt.yscale('log')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("\nKey Insight: Doubling precision requires 4x the data!")
```
</details>

## Reflection

Questions:
1. Why is standard error smaller than standard deviation?
2. What happens if sample size increases?
3. Why is this important in ML?


---

# Section 9: The Bootstrap

## What is Bootstrap?

Bootstrap is one of the most useful ideas in modern statistics. Instead of collecting new data, we repeatedly sample from existing data with replacement.

This helps estimate:
- Uncertainty
- Variability
- Confidence intervals

## Bootstrap Algorithm for Mean
1. Draw a sample value from data, record it, replace it
2. Repeat n times (same size as original sample)
3. Record the statistic of this resample
4. Repeat steps 1-3 R times (e.g., R=1000)
5. Use the R bootstrap statistics to estimate standard error and construct confidence intervals

<details>
<summary>Click to view solution</summary>

```python
bootstrap_means = []

for _ in range(1000):
    bootstrap_sample = state.sample(frac=1, replace=True)
    bootstrap_means.append(bootstrap_sample["Population"].mean())

plt.figure(figsize=(10, 6))
sns.histplot(bootstrap_means, bins=30, kde=True)
plt.title("Bootstrap Distribution of Means")
plt.xlabel("Mean Population")
plt.show()

print(f"Bootstrap Mean: {np.mean(bootstrap_means):,.2f}")
print(f"Bootstrap Std: {np.std(bootstrap_means):,.2f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Generic bootstrap function
def bootstrap_statistic(data, statistic_func, R=1000, random_state=None):
    """
    Generic bootstrap function.
    
    Parameters:
    -----------
    data : array-like
        Original sample data
    statistic_func : callable
        Function to compute statistic (e.g., np.mean, np.median)
    R : int
        Number of bootstrap resamples
    random_state : int, optional
        For reproducibility
    
    Returns:
    --------
    bootstrap_stats : array
        Array of bootstrap statistic values
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    bootstrap_stats = []
    n = len(data)
    
    for _ in range(R):
        resample = np.random.choice(data, size=n, replace=True)
        bootstrap_stats.append(statistic_func(resample))
    
    return np.array(bootstrap_stats)

# Load or simulate loan income data
try:
    loans_income = load_book_data('loans_income.csv')['income']
except:
    loans_income = np.random.lognormal(mean=10, sigma=1, size=5000)

# Take a sample
np.random.seed(42)
sample_loan = np.random.choice(loans_income, size=100, replace=False)

print(f"Original Sample (n={len(sample_loan)}):")
print(f"  Mean: ${np.mean(sample_loan):,.2f}")
print(f"  Median: ${np.median(sample_loan):,.2f}")
print(f"  SD: ${np.std(sample_loan):,.2f}")

# Bootstrap the median (robust to skew)
bootstrap_medians = bootstrap_statistic(sample_loan, np.median, R=2000, random_state=42)

# Calculate bootstrap SE and CI
bootstrap_se = np.std(bootstrap_medians)
ci_lower = np.percentile(bootstrap_medians, 2.5)
ci_upper = np.percentile(bootstrap_medians, 97.5)

print(f"\nBootstrap Results (R=2000):")
print(f"  Bootstrap SE: ${bootstrap_se:,.2f}")
print(f"  95% CI: [${ci_lower:,.2f}, ${ci_upper:,.2f}]")

# Visualize bootstrap distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(bootstrap_medians, bins=40, density=True, alpha=0.7, edgecolor='black')
axes[0].axvline(np.median(sample_loan), color='blue', linestyle='--', label=f'Original median: ${np.median(sample_loan):,.0f}')
axes[0].axvline(ci_lower, color='red', linestyle=':', label='95% CI bounds')
axes[0].axvline(ci_upper, color='red', linestyle=':')
axes[0].set_xlabel('Bootstrap Median')
axes[0].set_ylabel('Density')
axes[0].set_title('Bootstrap Distribution of Median')
axes[0].legend()
axes[0].grid(alpha=0.3)

# QQ-plot to check normality
stats.probplot(bootstrap_medians, dist="norm", plot=axes[1])
axes[1].set_title('QQ-Plot: Bootstrap Medians vs Normal')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nML Relevance: Bootstrap is used in Random Forests (bagging) to estimate model uncertainty!")
```
</details>

## Reflection

Questions:
1. Why does bootstrap work?
2. Why sample with replacement?
3. Where could bootstrap help in ML?



---

# Section 10: Confidence Intervals

## What is a confidence interval?

A confidence interval gives a plausible range for a population parameter.

**Correct interpretation**: "If we repeated this sampling procedure many times, 95% of such intervals would contain the true parameter"

**Incorrect interpretation**: "There is a 95% probability the true parameter lies in this interval"

## Factors Affecting Interval Width
- Higher confidence level → wider interval
- Smaller sample size → wider interval
- Greater data variability → wider interval

<details>
<summary>Click to view solution</summary>

```python
confidence_interval = np.percentile(bootstrap_means, [2.5, 97.5])

print("95% Confidence Interval")
print(f"Lower Bound: {confidence_interval[0]:,.2f}")
print(f"Upper Bound: {confidence_interval[1]:,.2f}")

plt.figure(figsize=(10, 6))
sns.histplot(bootstrap_means, bins=30, kde=True)
plt.axvline(confidence_interval[0], linestyle="--")
plt.axvline(confidence_interval[1], linestyle="--")
plt.title("95% Confidence Interval")
plt.show()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Compare CI methods: Bootstrap vs. Formula-Based
def bootstrap_ci(data, statistic=np.mean, confidence=0.95, R=2000, random_state=None):
    """Calculate bootstrap confidence interval."""
    if random_state is not None:
        np.random.seed(random_state)
    
    bootstrap_stats = bootstrap_statistic(data, statistic, R=R)
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_stats, 100 * alpha/2)
    upper = np.percentile(bootstrap_stats, 100 * (1 - alpha/2))
    
    return {
        'estimate': statistic(data),
        'se': np.std(bootstrap_stats),
        'ci_lower': lower,
        'ci_upper': upper,
        'bootstrap_stats': bootstrap_stats
    }

# Take a sample
np.random.seed(42)
sample_ci = np.random.choice(loans_income, size=100, replace=False)

# Method 1: Bootstrap CI for mean
bootstrap_mean_ci = bootstrap_ci(sample_ci, np.mean, R=2000, random_state=42)

# Method 2: Formula-based CI for mean (assuming normality)
se_formula = np.std(sample_ci, ddof=1) / np.sqrt(len(sample_ci))
t_critical = t.ppf(0.975, df=len(sample_ci)-1)
formula_ci_lower = np.mean(sample_ci) - t_critical * se_formula
formula_ci_upper = np.mean(sample_ci) + t_critical * se_formula

# Method 3: Bootstrap CI for median (no formula available!)
bootstrap_median_ci = bootstrap_ci(sample_ci, np.median, R=2000, random_state=42)

# Display results
print("Confidence Intervals (95%) for Loan Income Sample (n=100):")
print(f"{'Method':<25} {'Estimate':>12} {'SE':>10} {'95% CI Lower':>15} {'95% CI Upper':>15}")
print("-" * 80)
print(f"{'Bootstrap Mean':<25} ${bootstrap_mean_ci['estimate']:>10,.0f} ${bootstrap_mean_ci['se']:>8,.0f} ${bootstrap_mean_ci['ci_lower']:>13,.0f} ${bootstrap_mean_ci['ci_upper']:>13,.0f}")
print(f"{'Formula Mean (t-dist)':<25} ${np.mean(sample_ci):>10,.0f} ${se_formula:>8,.0f} ${formula_ci_lower:>13,.0f} ${formula_ci_upper:>13,.0f}")
print(f"{'Bootstrap Median':<25} ${bootstrap_median_ci['estimate']:>10,.0f} ${bootstrap_median_ci['se']:>8,.0f} ${bootstrap_median_ci['ci_lower']:>13,.0f} ${bootstrap_median_ci['ci_upper']:>13,.0f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compare mean CIs
axes[0].errorbar([1, 2], [bootstrap_mean_ci['estimate'], np.mean(sample_ci)],
                 yerr=[[bootstrap_mean_ci['estimate'] - bootstrap_mean_ci['ci_lower'],
                        np.mean(sample_ci) - formula_ci_lower],
                       [bootstrap_mean_ci['ci_upper'] - bootstrap_mean_ci['estimate'],
                        formula_ci_upper - np.mean(sample_ci)]],
                 fmt='o', capsize=8)
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(['Bootstrap', 'Formula'])
axes[0].set_ylabel('Mean Income ($)')
axes[0].set_title('95% Confidence Intervals for Mean')
axes[0].grid(alpha=0.3, axis='y')

# Bootstrap median distribution with CI
axes[1].hist(bootstrap_median_ci['bootstrap_stats'], bins=40, density=True, alpha=0.7, edgecolor='black')
axes[1].axvline(bootstrap_median_ci['estimate'], color='blue', linestyle='--', label='Estimate')
axes[1].axvline(bootstrap_median_ci['ci_lower'], color='red', linestyle=':', label='95% CI')
axes[1].axvline(bootstrap_median_ci['ci_upper'], color='red', linestyle=':')
axes[1].set_xlabel('Bootstrap Median')
axes[1].set_ylabel('Density')
axes[1].set_title('Bootstrap Distribution: Median')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight: Bootstrap works for ANY statistic, even when no formula exists!")
```
</details>

## Reflection

Questions:
1. Why is an interval better than a single number?
2. What happens with larger sample sizes?
3. Why do businesses care about uncertainty?

---



# Section 11: Normal Distribution

## What is a normal distribution?

A normal distribution is bell-shaped and symmetric.

Characteristics:
- Mean ≈ median ≈ mode
- Predictable probabilities
- 68% of data within ±1σ, 95% within ±2σ
- Symmetric shape

## Standard Normal Distribution
- μ = 0, σ = 1
- Values expressed as z-scores: z = (x - μ) / σ

<details>
<summary>Click to view solution</summary>

```python
normal_data = np.random.normal(loc=50, scale=10, size=10000)

plt.figure(figsize=(10, 6))
sns.histplot(normal_data, kde=True)
plt.title("Normal Distribution")
plt.show()

print(f"Mean: {np.mean(normal_data):.2f}")
print(f"Median: {np.median(normal_data):.2f}")
print(f"Std Dev: {np.std(normal_data):.2f}")
```
</details>

### QQ-Plots (Quantile-Quantile Plots)

Visual tool to assess if sample follows specified distribution. Points near diagonal line indicate good fit to distribution.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Generate samples from different distributions
distributions = {
    'Normal': np.random.normal(0, 1, 200),
    'Exponential': np.random.exponential(1, 200),
    'Uniform': np.random.uniform(-2, 2, 200),
    't-distribution (df=3)': stats.t.rvs(df=3, size=200)
}

# Create QQ-plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, data) in enumerate(distributions.items()):
    axes[idx].hist(data, bins=25, density=True, alpha=0.6, edgecolor='black', label='Sample')
    
    x = np.linspace(data.min(), data.max(), 100)
    axes[idx].plot(x, norm.pdf(x, np.mean(data), np.std(data)), 'r-', linewidth=2, label='Normal fit')
    
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Density')
    axes[idx].set_title(f'{name}\nSkewness: {stats.skew(data):.2f}, Kurtosis: {stats.kurtosis(data):.2f}')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# QQ-plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, data) in enumerate(distributions.items()):
    stats.probplot(data, dist="norm", plot=axes[idx])
    axes[idx].set_title(f'QQ-Plot: {name}')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate normality test p-values
print("\nShapiro-Wilk Test for Normality (p > 0.05 suggests normality):")
for name, data in distributions.items():
    stat_val, p_value = stats.shapiro(data)
    result = "Normal" if p_value > 0.05 else "Not Normal"
    print(f"  {name:<25}: p={p_value:.4f} -> {result}")

print("\nML Relevance: Many models assume normal residuals; QQ-plots help diagnose violations!")
```
</details>

## Reflection

Questions:
1. Why does this look symmetric?
2. Why do many statistical methods assume normality?
3. Are real-world datasets always normal?



---

# Section 12: Long-tail Distributions

## Real-world data is messy

Many datasets are NOT normal. Examples:
- Wealth
- Airline delays
- Fraud amounts
- Customer spending

These often follow long-tail distributions: many small values, few extreme values.

<details>
<summary>Click to view solution</summary>

```python
long_tail = np.random.exponential(scale=2, size=10000)

plt.figure(figsize=(10, 6))
sns.histplot(long_tail, bins=50, kde=True)
plt.title("Long-tail Distribution")
plt.show()

print("Mean:", np.mean(long_tail))
print("Median:", np.median(long_tail))
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Compare normal vs skewed distributions
normal_distribution = np.random.normal(50, 10, 10000)
skewed_distribution = np.random.exponential(2, 10000)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(normal_distribution, kde=True, ax=ax[0])
ax[0].set_title("Normal Distribution")

sns.histplot(skewed_distribution, kde=True, ax=ax[1])
ax[1].set_title("Long-tail Distribution")

plt.show()
```
</details>

## Reflection

Questions:
1. Why is mean larger than median?
2. Why are long-tail distributions common?
3. Why might median be safer here?
4. Which looks more realistic for business data?



---

# Section 13: Student's t-Distribution

## Characteristics
- Similar shape to normal but with thicker tails
- Family of distributions indexed by degrees of freedom (df)
- As df → ∞, t-distribution approaches normal
- Used when population standard deviation is unknown

<details>
<summary>Click to view solution</summary>

```python
# Plot t-distributions with different df vs. standard normal
dfs = [1, 3, 10, 30]
x = np.linspace(-4, 4, 1000)

plt.figure(figsize=(10, 6))

# Plot standard normal
plt.plot(x, norm.pdf(x), 'k-', linewidth=2.5, label='Standard Normal (df=inf)')

# Plot t-distributions
colors = ['red', 'orange', 'green', 'blue']
for df, color in zip(dfs, colors):
    plt.plot(x, t.pdf(x, df=df), color=color, linewidth=2, label=f't-distribution (df={df})')

plt.xlabel('x')
plt.ylabel('Density')
plt.title('t-Distribution vs. Normal: Effect of Degrees of Freedom')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Show critical values comparison
print("\nCritical Values for 95% Confidence (two-tailed):")
print(f"{'df':<5} {'t-critical':>12} {'Normal z':>12} {'Difference':>12}")
print("-" * 45)
for df in [1, 2, 5, 10, 30, 100]:
    t_crit = t.ppf(0.975, df=df)
    z_crit = norm.ppf(0.975)
    print(f"{df:<5} {t_crit:>12.3f} {z_crit:>12.3f} {abs(t_crit - z_crit):>12.3f}")

print(f"\nKey Insight: For small samples, t-distribution gives wider CIs (more conservative)!")
```
</details>


---

# Section 14: Binomial Distribution

## Applies to: Binary outcomes (success/failure, 1/0, yes/no)

### Parameters
- n = number of trials
- p = probability of success on each trial

### Key Properties
- Mean = np
- Variance = np(1-p)

### Data Science Applications
- Click-through rates, conversion rates, fraud detection
- A/B test analysis with binary outcomes

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Scenario: Email campaign with 2% click-through rate
p = 0.02  # probability of click
n = 500   # number of emails sent

# Calculate probabilities
k_values = np.arange(0, 25)
probabilities = binom.pmf(k_values, n=n, p=p)

# Expected value and variance
expected = n * p
variance = n * p * (1 - p)

print(f"Binomial Distribution: n={n}, p={p}")
print(f"  Expected clicks: {expected:.1f}")
print(f"  Standard deviation: {np.sqrt(variance):.2f}")
print(f"  P(0 clicks): {binom.pmf(0, n, p):.3f}")
print(f"  P(>=10 clicks): {1 - binom.cdf(9, n, p):.3f}")

# Visualize PMF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PMF
axes[0].bar(k_values, probabilities, alpha=0.7, edgecolor='black')
axes[0].axvline(expected, color='red', linestyle='--', label=f'Expected: {expected:.1f}')
axes[0].set_xlabel('Number of Clicks')
axes[0].set_ylabel('Probability')
axes[0].set_title('Binomial PMF: Clicks from 500 Emails')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# CDF
cdf_values = binom.cdf(k_values, n=n, p=p)
axes[1].plot(k_values, cdf_values, 'bo-', markersize=4)
axes[1].axhline(0.95, color='red', linestyle=':', label='95th percentile')
axes[1].axvline(binom.ppf(0.95, n, p), color='red', linestyle=':', label=f'95th percentile: {int(binom.ppf(0.95, n, p))}')
axes[1].set_xlabel('Number of Clicks')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('Binomial CDF')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Normal approximation check
if n * p > 5 and n * (1-p) > 5:
    print(f"\nNormal approximation valid (np={n*p:.1f} > 5, n(1-p)={n*(1-p):.1f} > 5)")
    mu, sigma = n*p, np.sqrt(n*p*(1-p))
    print(f"  Approximate with N(mu={mu:.1f}, sigma={sigma:.2f})")
else:
    print(f"\nNormal approximation NOT recommended (np={n*p:.1f} or n(1-p)={n*(1-p):.1f} <= 5)")

print("\nML Relevance: Binomial models underpin logistic regression and A/B testing!")
```
</details>


---

# Section 15: Poisson & Exponential Distributions

### Other Important Distributions

| Distribution | Use Case | Key Parameters |
|-------------|----------|---------------|
| **Chi-Square** | Testing independence; goodness-of-fit | Degrees of freedom |
| **F-Distribution** | ANOVA; comparing variances | Numerator df, denominator df |
| **Poisson** | Count data; events per unit time/space | lambda = rate |
| **Exponential** | Time between events in Poisson process | lambda = rate |

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Scenario: Website receives average 3 visits per hour (Poisson process)
lambda_rate = 3  # average events per time unit

# Poisson: Number of visits in one hour
poisson_visits = poisson.rvs(mu=lambda_rate, size=1000)

# Exponential: Time between visits (in hours)
exponential_times = expon.rvs(scale=1/lambda_rate, size=1000)

print(f"Poisson Distribution (lambda={lambda_rate}):")
print(f"  Mean visits/hour: {np.mean(poisson_visits):.2f} (theoretical: {lambda_rate})")
print(f"  P(0 visits): {poisson.pmf(0, lambda_rate):.3f}")
print(f"  P(>=5 visits): {1 - poisson.cdf(4, lambda_rate):.3f}")

print(f"\nExponential Distribution (lambda={lambda_rate}):")
print(f"  Mean time between visits: {np.mean(exponential_times):.2f} hours (theoretical: {1/lambda_rate:.2f})")
print(f"  P(wait < 0.5 hours): {expon.cdf(0.5, scale=1/lambda_rate):.3f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Poisson PMF
k_poisson = np.arange(0, 12)
axes[0].bar(k_poisson, poisson.pmf(k_poisson, lambda_rate), alpha=0.7, edgecolor='black', label='Theoretical')
axes[0].hist(poisson_visits, bins=np.arange(-0.5, 11.5), density=True, alpha=0.5, label='Simulated', edgecolor='black')
axes[0].set_xlabel('Visits per Hour')
axes[0].set_ylabel('Probability')
axes[0].set_title(f'Poisson Distribution (lambda={lambda_rate})')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Exponential PDF
x_exp = np.linspace(0, 3, 100)
axes[1].plot(x_exp, expon.pdf(x_exp, scale=1/lambda_rate), 'r-', linewidth=2, label='Theoretical PDF')
axes[1].hist(exponential_times, bins=30, density=True, alpha=0.5, label='Simulated', edgecolor='black')
axes[1].set_xlabel('Time Between Visits (hours)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Exponential Distribution (lambda={lambda_rate})')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nML Relevance: Poisson regression for count outcomes; exponential for survival analysis!")
```
</details>



---

# Section 16: Comparing Sample Sizes

How does sample size affect reliability? Larger samples should produce more stable estimates.

<details>
<summary>Click to view solution</summary>

```python
sample_sizes = [5, 10, 30, 50]
means_dict = {}

for n in sample_sizes:
    means = []
    for _ in range(1000):
        sample = state.sample(n=n, replace=True)
        means.append(sample["Population"].mean())
    means_dict[n] = means

fig, ax = plt.subplots(2, 2, figsize=(14, 10))
ax = ax.flatten()

for i, n in enumerate(sample_sizes):
    sns.histplot(means_dict[n], bins=30, kde=True, ax=ax[i])
    ax[i].set_title(f"Sample Size = {n}")

plt.tight_layout()
plt.show()
```
</details>



---

# Section 17: Sampling Bias Experiment

Compare random sample vs biased sample.

<details>
<summary>Click to view solution</summary>

```python
random_sample_20 = state.sample(n=20, random_state=42)
biased_sample_20 = state.nlargest(20, "Population")

comparison = pd.DataFrame({
    "Population Mean": [
        state["Population"].mean(),
        random_sample_20["Population"].mean(),
        biased_sample_20["Population"].mean()
    ]
}, index=["Population", "Random Sample", "Biased Sample"])

comparison
```
</details>

<details>
<summary>Click to view solution</summary>

```python
comparison.plot(kind="bar", figsize=(8, 5))
plt.title("Population vs Sample Means")
plt.ylabel("Population Mean")
plt.show()
```
</details>



---

# Section 18: Bootstrap Confidence Interval Experiment

How stable are confidence intervals? Run bootstrap repeatedly.

<details>
<summary>Click to view solution</summary>

```python
confidence_intervals = []

for _ in range(100):
    sample = state.sample(frac=1, replace=True)
    mean_value = sample["Population"].mean()
    confidence_intervals.append(mean_value)

plt.figure(figsize=(10, 6))
sns.histplot(confidence_intervals, bins=25, kde=True)
plt.title("Bootstrap Mean Stability")
plt.show()
```
</details>



---

# Mini Project: Analyse Sampling Behaviour

## Goal

Investigate how stable sample estimates are.

### Tasks
1. Create 100 samples of size 10
2. Calculate sample means
3. Plot distribution
4. Compare with true population mean
5. Write interpretation

<details>
<summary>Click to view solution</summary>

```python
mini_project_means = []

for _ in range(100):
    sample = state.sample(n=10, replace=True)
    mini_project_means.append(sample["Population"].mean())

plt.figure(figsize=(10, 6))
sns.histplot(mini_project_means, bins=25, kde=True)
plt.axvline(state["Population"].mean(), linestyle="--", label="Population Mean")
plt.legend()
plt.title("Mini Project: Sample Means")
plt.show()
```
</details>

## Interpretation

1. What did you observe?
2. Were means identical?
3. Why is sampling uncertainty important?


---

# ML Relevance: Why Chapter 2 Matters

## Sampling = Machine Learning

Machine learning never sees entire reality — it sees samples. Training data is sampled reality.

## Bias = Bad ML

Biased datasets create biased predictions, unfair systems, poor generalisation. Examples: facial recognition bias, recommendation bias, hiring bias.

## Bootstrap = Ensemble Learning

Random forests use bootstrap sampling internally.

## Confidence Intervals

Useful for model evaluation, uncertainty estimation, A/B testing.

## Distribution Awareness

Many features require log transform, scaling, or standardisation before modelling.

---

# Interview Questions

1. **Difference between population and sample?**
2. **What is sampling bias?**
3. **Explain Central Limit Theorem.**
4. **Difference between standard deviation and standard error?**
5. **What is bootstrap sampling?**
6. **What is a confidence interval?**
7. **Why are real-world datasets often skewed?**
8. **Why does train/test split matter?**
9. **What is regression to the mean?**
10. **When would you use t-distribution vs normal distribution?**

---

# Challenge Problems

1. Create a simulation showing sample size vs standard error
2. Create biased vs unbiased sampling experiment
3. Simulate 1000 bootstrap confidence intervals
4. Compare normal distribution vs skewed distribution and explain implications for ML
5. Demonstrate regression to the mean with a real-world scenario

---

# Chapter Summary

## Key ideas learned
- Population vs sample
- Random sampling and sampling bias
- Regression to the mean
- Sampling variability and distributions
- Central Limit Theorem
- Standard error
- Bootstrap
- Confidence intervals
- Normal, t, binomial, Poisson, and exponential distributions

## Biggest takeaway

Most of statistics is learning how to make reliable conclusions from incomplete data.

---

## ML Relevance Recap

| Concept | ML Application |
|---------|---------------|
| **Random Sampling** | Train/test splits, cross-validation, stratified sampling |
| **Bootstrap** | Random forests (bagging), model uncertainty estimation |
| **Confidence Intervals** | Uncertainty in predictions, A/B test results |
| **CLT** | Justifies normal approximations for large-sample inference |
| **Distribution Choice** | GLMs, count models, survival analysis |

---

## Save Processed Data

<details>
<summary>Click to view solution</summary>

```python
# Save processed data for later chapters
output_path = '../datasets/processed/chapter2_bootstrap_demo.pkl'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

import pickle
processed_data = {
    'bootstrap_medians': bootstrap_medians,
    'bootstrap_means': bootstrap_means,
    'population_mean': np.mean(loans_income),
    'sample_size': len(sample_loan)
}

with open(output_path, 'wb') as f:
    pickle.dump(processed_data, f)

print(f"Saved bootstrap demo data to {output_path}")

# Save loans_income if we simulated it
if not os.path.exists('../datasets/raw/loans_income.csv'):
    pd.DataFrame({'income': loans_income}).to_csv('../datasets/raw/loans_income.csv', index=False)
    print("Saved simulated loans_income.csv to ../datasets/raw/")
```
</details>

---

# Progress Checklist

- [ ] Understood population vs sample
- [ ] Learned random sampling
- [ ] Identified sampling bias and selection bias
- [ ] Understood regression to the mean
- [ ] Distinguished data distribution vs sampling distribution
- [ ] Visualised Central Limit Theorem
- [ ] Calculated and interpreted standard error
- [ ] Implemented bootstrap sampling
- [ ] Built and interpreted confidence intervals
- [ ] Analysed normal distribution and QQ-plots
- [ ] Explored t-distribution vs normal
- [ ] Applied binomial distribution for binary outcomes
- [ ] Used Poisson and exponential for event data
- [ ] Compared sample sizes
- [ ] Completed sampling bias experiment
- [ ] Completed mini project
- [ ] Understood ML relevance
- [ ] Answered interview questions
- [ ] Ready for Chapter 3

---